# MCP Server Tools Explorer

Retrieve and display available tools from MCP (Model Context Protocol) servers configured in `servers.json`.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv(override=True)


In [ ]:
url  = 'https://order-mcp-74afyau24q-uc.a.run.app/mcp'
init_msg = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "my-client", "version": "1.0.0"}
    }
}
res = requests.post(url, json=init_msg, headers={"Accept": "application/json, text/event-stream"}, stream=True)
for line in res.iter_lines():
    if line:
        decoded = line.decode("utf-8")
        if decoded.startswith("data: "):
            print(json.loads(decoded[6:]))
        else:
            print(decoded)

{"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-03-26","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"order-mcp","version":"1.22.0"}}}


In [ ]:
_request_id = 2
_session_id = None
_player_token = None

def call_tool(tool_name, arguments=None):
    """Call an MCP tool and return the parsed result."""
    global _request_id, _session_id

    _request_id += 1
    msg = {
        "jsonrpc": "2.0",
        "id": _request_id,
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": arguments or {}
        }
    }

    headers = {"Accept": "application/json, text/event-stream"}
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    if _player_token:
        headers["x-player-token"] = _player_token

    res = requests.post(url, json=msg, headers=headers, stream=True)

    if sid := res.headers.get("Mcp-Session-Id"):
        _session_id = sid

    for line in res.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8")
        if not decoded.startswith("data: "):
            continue

        data = json.loads(decoded[6:])
        if "error" in data:
            raise RuntimeError(f"MCP error: {data['error']}")
        if "result" not in data:
            return data

        content = data["result"].get("content", [])
        texts = [c["text"] for c in content if c.get("type") == "text"]
        if not texts:
            return content

        try:
            return json.loads(texts[0])
        except (json.JSONDecodeError, TypeError):
            return texts[0] if len(texts) == 1 else texts

    return None

In [ ]:
def _read_json_or_sse(res):
    if "application/json" in (res.headers.get("content-type") or ""):
        return res.json()
    for line in res.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8")
        if decoded.startswith("data: "):
            return json.loads(decoded[6:])
    return None

headers = {"Accept": "application/json, text/event-stream"}

res = requests.post(
    url,
    json={"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}},
    headers=headers,
)
data = _read_json_or_sse(res)
tools = data.get("result", {}).get("tools", []) if data else []

print(f"{len(tools)} tools available on order-mcp:\n")
for t in tools:
    desc = (t.get("description") or "").strip().splitlines()[0] if t.get("description") else ""
    print(f"- {t['name']}: {desc}")


8 tools available on order-mcp:

- list_products: List products with optional filters.
- get_product: Get detailed product information by SKU.
- search_products: Search products by name or description.
- get_customer: Get customer information by ID.
- verify_customer_pin: Verify customer identity with email and PIN.
- list_orders: List orders with optional filters.
- get_order: Get detailed order information including items.
- create_order: Create a new order with items.


## Chatbot demo

Cells below boot the FastAPI service as a subprocess (so the whole notebook is self-contained), then exercise the `/v1/chat` endpoint.

Requires in `.env`:
- `ANDELA_MCP_GROQ_API_KEY` (mandatory — chat agent inference goes to Groq)
- `ANDELA_MCP_OPENAI_API_KEY` (optional — Agents SDK trace export to the OpenAI dashboard)

**Test customers** (email / PIN — seeded fixtures in the order-mcp upstream):

| email | pin |
|---|---|
| donaldgarcia@example.net | 7912 |
| michellejames@example.com | 1520 |
| laurahenderson@example.org | 1488 |
| spenceamanda@example.org | 2535 |
| glee@example.net | 4582 |
| williamsthomas@example.net | 4811 |
| justin78@example.net | 9279 |
| jason31@example.com | 1434 |
| samuel81@example.com | 4257 |
| williamleon@example.net | 9928 |

In [8]:
# Start the FastAPI service as a background subprocess.
# Re-running this cell tears down any prior instance first.
import subprocess, sys, time, atexit, signal, pathlib, socket

# Edit these directly — they take precedence over .env / shell env vars.
PORT = 8084
LLM_MODEL = "openai/gpt-oss-120b:free"   # use ":free" until you've added OpenRouter credits

BASE_URL = f"http://127.0.0.1:{PORT}"
PROJECT_ROOT = pathlib.Path("/Users/chrys/Ai_engineering/andela-mcp")

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

# Stop any prior server we launched from this notebook
proc = globals().get("server_proc")
if proc and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=5)
    except subprocess.TimeoutExpired: proc.kill()

if _port_in_use(PORT):
    raise RuntimeError(f"port {PORT} already in use — change PORT above to a free one")

env = {
    **os.environ,
    "ANDELA_MCP_PORT": str(PORT),
    "ANDELA_MCP_LLM_MODEL": LLM_MODEL,
    "ANDELA_MCP_LOG_FORMAT": "console",
}
server_proc = subprocess.Popen(
    [sys.executable, "-m", "andela_mcp"],
    cwd=str(PROJECT_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
atexit.register(lambda: server_proc.poll() is None and server_proc.terminate())

# Wait for /healthz
deadline = time.time() + 30
while time.time() < deadline:
    if server_proc.poll() is not None:
        print(server_proc.stdout.read() if server_proc.stdout else "")
        raise RuntimeError(f"server exited early with code {server_proc.returncode}")
    try:
        if requests.get(f"{BASE_URL}/healthz", timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(0.5)
else:
    raise RuntimeError("server did not become ready within 30s")

print(f"server running pid={server_proc.pid} at {BASE_URL}  (model={LLM_MODEL})")
print("MCP servers:", requests.get(f"{BASE_URL}/readyz").json().get("servers"))

server running pid=49732 at http://127.0.0.1:8084  (model=openai/gpt-oss-120b:free)
MCP servers: ['remote-mcp']


In [9]:
CHAT_URL = f"{BASE_URL}/v1/chat"
conversation = []

def chat(user_msg):
    conversation.append({"role": "user", "content": user_msg})
    res = requests.post(CHAT_URL, json={"messages": conversation}, timeout=180)
    if not res.ok:
        try:
            detail = res.json().get("detail", res.text)
        except ValueError:
            detail = res.text
        raise RuntimeError(f"{res.status_code} from {CHAT_URL}: {detail}")
    body = res.json()
    conversation.append({"role": "assistant", "content": body["reply"]})
    print(f"USER: {user_msg}")
    for tc in body.get("tool_calls", []):
        print(f"  -> {tc['server']}.{tc['tool']}({tc['arguments']})")
    print(f"BOT:  {body['reply']}\n")
    return body

# Test 1 — browse (no auth needed)
chat("What product categories do you sell?")

# Test 2 — auth + customer-scoped query (single turn, multi-tool)
conversation.clear()
chat("Hi, I'm donaldgarcia@example.net, my PIN is 7912. Verify me, then list my orders.")

# Test 3 — follow-up turn (uses prior conversation as context)
chat("Show me the items in the most recent draft order.")

USER: What product categories do you sell?
  -> remote-mcp.list_products({'category': None, 'is_active': None})
BOT:  We offer a wide range of electronics divided into these main categories:

- **Computers** (desktops, laptops, all‑in‑ones, mini PCs, Chromebooks, MacBooks, etc.)  
- **Monitors** (24‑, 27‑, 32‑inch, 4K, ultrawide, curved, portable, touch‑screen)  
- **Printers** (laser, inkjet, all‑in‑one, photo, large‑format, 3D, label, receipt)  
- **Accessories** (keyboards, mice, gaming peripherals, webcams, headsets, USB hubs, docking stations, KVM switches, cable‑management kits)  
- **Networking** (wireless routers, switches, access points, modems)  

Let me know if you’d like details about a specific category or product!

USER: Hi, I'm donaldgarcia@example.net, my PIN is 7912. Verify me, then list my orders.
  -> remote-mcp.verify_customer_pin({'email': 'donaldgarcia@example.net', 'pin': '7912'})
  -> remote-mcp.list_orders({'customer_id': '41c2903a-f1a5-47b7-a81d-86b50ade220f'}

{'reply': 'Here are the items in your most recent **draft** order (ID\u202f2f624b70‑3227‑4632‑b6dc‑6848daa0b367):\n\n| SKU | Product | Qty | Unit Price | Subtotal |\n|-----|---------|-----|------------|----------|\n| ACC-0163 | USB Hub – Model\u202fC | 1 | $36.30 | $36.30 |\n| ACC-0143 | Wireless Mouse – Model\u202fC | 3 | $63.34 | $190.02 |\n| PRI-0109 | Photo Printer – Model\u202fD | 4 | $241.16 | $964.64 |\n| MON-0063 | 32‑inch Monitor – Model\u202fC | 1 | $800.30 | $800.30 |\n\n**Total:** $1,991.26\u202fUSD. Let me know if you’d like to edit this order, place a new one, or need anything else.',
 'tool_calls': [{'server': 'remote-mcp',
   'tool': 'list_orders',
   'arguments': {'status': 'draft'},
   'result': [{'type': 'text',
     'text': 'Found 5 orders:\n\nOrder ID: 2f624b70-3227-4632-b6dc-6848daa0b367\n  Status: draft | Payment: pending\n  Total: $1991.26 USD\n  Created: 2026-04-29\n\nOrder ID: 1628eb31-1aec-44fa-ab60-40c950d80334\n  Status: draft | Payment: pending\n  Total: $

In [ ]:
# Stop the server (run when you're done — or let kernel shutdown / atexit do it)
if server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
print("server stopped, returncode =", server_proc.returncode)